In [1]:
import numpy as np
import pandas as pd
import joblib

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, f1_score

import mlflow
import mlflow.sklearn

X_train, X_test, y_train, y_test = joblib.load('../data/processed/train_test_split.pkl')

print(f"Train shape: {X_train.shape}")
print(f"Test shape: {X_test.shape}")
print(f"Train class distribution: {y_train.value_counts()}")

Train shape: (800, 33)
Test shape: (200, 33)
Train class distribution: class
0    559
1    241
Name: count, dtype: int64


In [2]:
mlflow.set_experiment("credit-risk-baseline")
print("MLFLOW TRACKING URI:", mlflow.get_tracking_uri())

2026/03/20 12:54:48 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/03/20 12:54:48 INFO mlflow.store.db.utils: Updating database tables
2026/03/20 12:54:49 INFO mlflow.tracking.fluent: Experiment with name 'credit-risk-baseline' does not exist. Creating a new experiment.


MLFLOW TRACKING URI: sqlite:////home/purshotam_kumar/ml_projects/credit-risk-prediction/notebooks/mlflow.db


In [3]:
with mlflow.start_run(run_name="Logistic_Regression"):
		
		lr = LogisticRegression(max_iter=1000, random_state=42)
		lr.fit(X_train, y_train)

		y_pred = lr.predict(X_test)
		y_pred_proba = lr.predict_proba(X_test)[:, 1]

		f1 = f1_score(y_test, y_pred)
		roc_auc = roc_auc_score(y_test, y_pred_proba)

		mlflow.log_param("model_type", "Logistic Regression")
		mlflow.log_metric("f1_score", f1)
		mlflow.log_metric("roc_auc", roc_auc)
		mlflow.sklearn.log_model(lr, "model")

		print("Logistic Regression Results:")
		print(f"F1 Score: {f1:.4f}")
		print(f"ROC-AUC: {roc_auc:.4f}")
		print("\nClassification Report:")
		print(classification_report(y_test, y_pred))
		print("\nConfusion Matrix:")
		print(confusion_matrix(y_test, y_pred))

/home/purshotam_kumar/ml_projects/project-1/venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
2026/03/20 12:54:57 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/20 12:54:58 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The r

Logistic Regression Results:
F1 Score: 0.6111
ROC-AUC: 0.8096

Classification Report:
              precision    recall  f1-score   support

           0       0.83      0.89      0.86       141
           1       0.67      0.56      0.61        59

    accuracy                           0.79       200
   macro avg       0.75      0.72      0.73       200
weighted avg       0.78      0.79      0.78       200


Confusion Matrix:
[[125  16]
 [ 26  33]]


In [4]:
with mlflow.start_run(run_name="Random_Forest"):
  
  rf = RandomForestClassifier(n_estimators=100, random_state=42)
  rf.fit(X_train, y_train)
  
  y_pred = rf.predict(X_test)
  y_pred_proba = rf.predict_proba(X_test)[:, 1]
  
  f1 = f1_score(y_test, y_pred)
  roc_auc = roc_auc_score(y_test, y_pred_proba)
  mlflow.log_param("model_type", "Random Forest")
  mlflow.log_param("n_estimators", 100)
  mlflow.log_metric("f1_score", f1)
  mlflow.log_metric("roc_auc", roc_auc)
  mlflow.sklearn.log_model(rf, "model")

  print("Random Forest Results:")
  print(f"F1 Score: {f1:.4f}")
  print(f"ROC-AUC: {roc_auc:.4f}")
  print("\nClassification Report:")
  print(classification_report(y_test, y_pred))
  print("\nConfusion Matrix:")
  print(confusion_matrix(y_test, y_pred))

2026/03/20 12:56:13 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/20 12:56:13 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Random Forest Results:
F1 Score: 0.5600
ROC-AUC: 0.8211

Classification Report:
              precision    recall  f1-score   support

           0       0.81      0.91      0.85       141
           1       0.68      0.47      0.56        59

    accuracy                           0.78       200
   macro avg       0.74      0.69      0.71       200
weighted avg       0.77      0.78      0.77       200


Confusion Matrix:
[[128  13]
 [ 31  28]]


In [5]:
with mlflow.start_run(run_name="XGBoost"):
  
		xgb = XGBClassifier(n_estimators=100, use_label_encoder=False, eval_metric='logloss', random_state=42)
		xgb.fit(X_train, y_train)
		
		y_pred = xgb.predict(X_test)
		y_pred_proba = xgb.predict_proba(X_test)[:, 1]
		
		f1 = f1_score(y_test, y_pred)
		roc_auc = roc_auc_score(y_test, y_pred_proba)

		mlflow.log_param("model_type", "XGBoost")
		mlflow.log_param("n_estimators", 100)
		mlflow.log_metric("f1_score", f1)
		mlflow.log_metric("roc_auc", roc_auc)
		mlflow.sklearn.log_model(xgb, "model")

		print("XGBoost Results:")
		print(f"F1 Score: {f1:.4f}")
		print(f"ROC-AUC: {roc_auc:.4f}")
		print("\nClassification Report:")
		print(classification_report(y_test, y_pred))
		print("\nConfusion Matrix:")
		print(confusion_matrix(y_test, y_pred))

/home/purshotam_kumar/ml_projects/project-1/venv/lib/python3.12/site-packages/xgboost/training.py:200: UserWarning: [12:57:24] WARNING: /__w/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
2026/03/20 12:57:24 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/20 12:57:24 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


XGBoost Results:
F1 Score: 0.5946
ROC-AUC: 0.8077

Classification Report:
              precision    recall  f1-score   support

           0       0.82      0.87      0.84       141
           1       0.63      0.56      0.59        59

    accuracy                           0.78       200
   macro avg       0.73      0.71      0.72       200
weighted avg       0.77      0.78      0.77       200


Confusion Matrix:
[[122  19]
 [ 26  33]]


## Model Comparison Summary

**Best Model: XGBoost**
- F1 Score: 0.6727
- ROC-AUC: 0.8275
- Precision on bad credit (class 1): 0.73
- Recall on bad credit: 0.63

**Why XGBoost wins:**
- Catches more defaults (37 out of 59)
- Higher precision when predicting default (73%)
- Fewest false negatives (22 vs 25)
- Best overall accuracy (82%)

**Next steps:**
- Week 3: Hyperparameter tuning on XGBoost
- Add SHAP for interpretability
- Cost-sensitive threshold tuning

In [6]:
joblib.dump(xgb, '../models/xgb_model.pkl')
print("XGBoost model saved")

XGBoost model saved
